### Data Quality Checks
Load raw CSVs and validate completeness, data types, outliers, and categorical distributions.

In [14]:
import pandas as pd
from pathlib import Path

_cwd = Path.cwd()
DATA_RAW = _cwd / 'data' / 'raw' if (_cwd / 'data' / 'raw').exists() else _cwd.parent / 'data' / 'raw'

required_files = ['transactions.csv', 'users.csv', 'merchants.csv']
missing = [f for f in required_files if not (DATA_RAW / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing: {missing}. Run generator/transaction_generator.py first.')

txn = pd.read_csv(DATA_RAW / 'transactions.csv')
users = pd.read_csv(DATA_RAW / 'users.csv')
merchants = pd.read_csv(DATA_RAW / 'merchants.csv')

def quality_report(df, label):
    """Print data quality metrics: row/col counts, duplicates, and null statistics."""
    info = pd.DataFrame({
        'col': df.columns,
        'dtype': df.dtypes.values,
        'nulls': df.isnull().sum().values,
        'null%': (df.isnull().sum() / len(df) * 100).round(2).values
    })
    print(f'{label} ({len(df):,} rows, {len(df.columns)} cols)')
    print(f'Duplicates: {df.duplicated().sum()}')
    print(info.to_string(index=False))

quality_report(txn, 'TRANSACTIONS')
quality_report(users, 'USERS')
quality_report(merchants, 'MERCHANTS')

TRANSACTIONS (400,000 rows, 14 cols)
Duplicates: 0
              col   dtype  nulls  null%
   transaction_id  object      0    0.0
          user_id  object      0    0.0
      merchant_id  object      0    0.0
merchant_category  object      0    0.0
   payment_method  object      0    0.0
           amount float64      0    0.0
         currency  object      0    0.0
           status  object      0    0.0
         is_fraud    bool      0    0.0
      device_type  object      0    0.0
             city  object      0    0.0
            state  object      0    0.0
   transaction_ts  object      0    0.0
       created_at  object      0    0.0
USERS (1,000 rows, 6 cols)
Duplicates: 0
              col  dtype  nulls  null%
          user_id object      0    0.0
        age_group object      0    0.0
     account_type object      0    0.0
registration_date object      0    0.0
             city object      0    0.0
            state object      0    0.0
MERCHANTS (200 rows, 6 cols)
Duplic

In [15]:
# Flag values far outside the middle 50% as outliers
Q1 = txn['amount'].quantile(0.25)
Q3 = txn['amount'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = txn[(txn['amount'] < lower) | (txn['amount'] > upper)]
print(f'Amount IQR: [{Q1:.2f}, {Q3:.2f}]')
print(f'Amount range: {txn["amount"].min():.2f} - {txn["amount"].max():.2f}')
print(f'Outliers: {len(outliers):,} ({len(outliers)/len(txn)*100:.2f}%)')

Amount IQR: [6444.80, 19339.59]
Amount range: 50.01 - 59996.81
Outliers: 9,906 (2.48%)


In [16]:
for col in ['payment_method', 'device_type', 'status', 'currency']:
    print(f'\n--- {col} ---')
    tab = pd.DataFrame({'category': txn[col].value_counts().index,
                        'count': txn[col].value_counts().values,
                        'pct': txn[col].value_counts(normalize=True).mul(100).round(2).values})
    print(tab.to_string(index=False))


--- payment_method ---
   category  count   pct
 NetBanking  80264 20.07
Credit Card  80134 20.03
 Debit Card  80019 20.00
     Wallet  79875 19.97
        UPI  79708 19.93

--- device_type ---
category  count   pct
  mobile 200614 50.15
 desktop  99918 24.98
     POS  99468 24.87

--- status ---
category  count   pct
 success 228617 57.15
  failed  57303 14.33
disputed  57190 14.30
declined  56890 14.22

--- currency ---
category  count   pct
     INR 400000 100.0


In [17]:
def cat_table(series, label):
    vc = series.value_counts()
    tab = pd.DataFrame({'category': vc.index, 'count': vc.values})
    print(f'\n--- {label} ---')
    print(tab.to_string(index=False))

cat_table(users['age_group'], 'age group')
cat_table(users['account_type'], 'account type')
cat_table(merchants['merchant_category'], 'merchant category')


--- age group ---
category  count
     50+    257
   26-35    251
   18-25    247
   36-50    245

--- account type ---
category  count
 current    358
 premium    331
 savings    311

--- merchant category ---
       category  count
Food & Beverage     32
         Retail     29
  Entertainment     27
         Travel     25
      Groceries     25
     Healthcare     21
        Fashion     21
    Electronics     20


In [18]:
fraud_rate = txn['is_fraud'].mean() * 100
fraud_count = txn['is_fraud'].sum()
print(f'Fraud rate: {fraud_rate:.2f}%')
print(f'Fraudulent transactions: {fraud_count:,} / {len(txn):,}')

Fraud rate: 3.49%
Fraudulent transactions: 13,970 / 400,000


In [19]:
has_nulls = txn.isnull().any().any() or users.isnull().any().any() or merchants.isnull().any().any()
has_dupes = txn.duplicated().any() or users.duplicated().any() or merchants.duplicated().any()
fraud_rate = txn['is_fraud'].mean() * 100
outlier_pct = len(outliers) / len(txn) * 100

print('Data Quality Summary')
print(f'Missing data: {"Yes" if has_nulls else "No"} -- {"cleaning required" if has_nulls else "no cleaning needed"}')
print(f'Duplicate records: {"Found" if has_dupes else "None"} -- IDs are {"not " if has_dupes else ""}unique')
print(f'Fraud rate: {fraud_rate:.2f}% -- realistic for payment systems')
print(f'Outliers: {outlier_pct:.2f}% -- high-value transactions beyond IQR bounds')

Data Quality Summary
Missing data: No -- no cleaning needed
Duplicate records: None -- IDs are unique
Fraud rate: 3.49% -- realistic for payment systems
Outliers: 2.48% -- high-value transactions beyond IQR bounds
